In [3]:
import json
from pathlib import Path
from collections import Counter, defaultdict

def scan_labels_only(base_dir):
    print(f"[{base_dir}] json스캔\n")
    
    json_paths = list(Path(base_dir).rglob("*.json"))
    
    if not json_paths:
        return

    print(f"총 {len(json_paths):,}개\n")


    
    label_counts = Counter()               
    label_to_nums = defaultdict(set)        
    
    total_normal_images = 0
    total_defect_images = 0


    for json_path in json_paths:
        with open(json_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                continue
            
            img_info = data.get("image", {})
            is_defect = img_info.get("object_included", "N")
            
            if is_defect == "N":
                total_normal_images += 1
            elif is_defect == "Y":
                total_defect_images += 1
                annotations = img_info.get("annotations", [])
                
                for ann in annotations:
                    lbl = ann.get("label")
                    lbl_num = ann.get("labelNum")
                    
                    if lbl:
                        label_counts[lbl] += 1
                        if lbl_num is not None:
                            label_to_nums[lbl].add(lbl_num)

    print("데이터 스캔 결과")
    print(f"균열 없음 이미지 수 : {total_normal_images:,}장")
    print(f"균열 있음 이미지 수 : {total_defect_images:,}장")
    print("결함 라벨 종류 및 통계")
    
    for lbl, count in label_counts.most_common():
        nums = sorted(list(label_to_nums[lbl]))
        nums_str = str(nums[:5]) + " ..." if len(nums) > 5 else str(nums)
        
        print(f" 라벨명: '{lbl}'")
        print(f"  폴리곤(조각) 수: {count:,}개")
        print(f"  발견된 labelNum: {nums_str}\n")

In [5]:
BASE_DIR = "D:\Study\HumanStudy\Dataset\Validation"
scan_labels_only(BASE_DIR)

[D:\Study\HumanStudy\Dataset\Validation] json스캔

총 52,500개

데이터 스캔 결과
균열 없음 이미지 수 : 2,500장
균열 있음 이미지 수 : 50,000장
결함 라벨 종류 및 통계
 라벨명: 'crack'
  폴리곤(조각) 수: 128,279개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'reticular crack'
  폴리곤(조각) 수: 19,303개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'efflorescence'
  폴리곤(조각) 수: 4,619개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'detachment'
  폴리곤(조각) 수: 3,702개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'leak'
  폴리곤(조각) 수: 3,194개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'spalling'
  폴리곤(조각) 수: 2,723개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'material separation'
  폴리곤(조각) 수: 2,014개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'rebar'
  폴리곤(조각) 수: 446개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'damage'
  폴리곤(조각) 수: 377개
  발견된 labelNum: [0, 1, 2, 3, 4] ...

 라벨명: 'exhilaration'
  폴리곤(조각) 수: 139개
  발견된 labelNum: [0, 1, 2, 3, 4] ...



In [8]:
import os
import json
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm

def deep_eda_json(base_dir):
    
    json_paths = list(Path(base_dir).rglob("*.json"))
        
    total_images = 0
    normal_images = 0
    defect_images = 0
    
    max_polygons_in_one_image = 0
    max_polygons_image_name = ""
    
    # 이미지당 결함 종류의 개수
    types_per_image_count = Counter() 

    # 결함 조합의 수
    defect_combinations = Counter()
    
    # 클래스별로 등장한 이미지 수
    images_per_class = Counter()

    for json_path in tqdm(json_paths):
        with open(json_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                continue
                
            img_info = data.get("image", {})
            is_defect = img_info.get("object_included", "N")
            total_images += 1
            
            if is_defect == "N":
                normal_images += 1
            elif is_defect == "Y":
                defect_images += 1
                annotations = img_info.get("annotations", [])
                
                num_polygons = len(annotations)
                if num_polygons > max_polygons_in_one_image:
                    max_polygons_in_one_image = num_polygons
                    max_polygons_image_name = img_info.get("name", "Unknown")
                
                unique_labels = set(ann.get("label", "").lower() for ann in annotations if ann.get("label"))
                
                types_per_image_count[len(unique_labels)] += 1
                
                combo = tuple(sorted(unique_labels))
                defect_combinations[combo] += 1
                
                for label in unique_labels:
                    images_per_class[label] += 1

    print(f"전체 이미지 수 : {total_images:,}장")
    print(f"   정상 이미지 : {normal_images:,}장")
    print(f"   결함 이미지 : {defect_images:,}장")
    
    print(f"\n한 이미지 내 최대 결함(Polygon) 개수")
    print(f"   최대 {max_polygons_in_one_image:,}개의 결함 조각이 한 사진에 존재함.")
    print(f"   해당 파일명: {max_polygons_image_name}")
    
    print(f"\n한 이미지 내 결함 종류 섞임 정도 ")
    for num_types, count in sorted(types_per_image_count.items()):
        ratio = (count / defect_images) * 100
        print(f"  {num_types}종류의 결함이 섞인 이미지: {count:,}장 ({ratio:.1f}%)")
    
    print(f"\n가장 많이 등장한 결함 조합 ")
    for combo, count in defect_combinations.most_common(5):
        combo_str = " + ".join(combo)
        print(f"   [{combo_str}] 조합 : {count:,}장")
    
    print(f"\n최종 클래스별 이미지 수")
    for label, count in images_per_class.most_common():
        print(f"   {label} : {count:,}장")

TRAIN_BASE_DIR = r"D:\Study\HumanStudy\Dataset\Validation"
deep_eda_json(TRAIN_BASE_DIR)

100%|███████████████████████████████████████████████████████████████████████████| 52500/52500 [06:49<00:00, 128.36it/s]

전체 이미지 수 : 52,500장
   정상 이미지 : 2,500장
   결함 이미지 : 50,000장

한 이미지 내 최대 결함(Polygon) 개수
   최대 58개의 결함 조각이 한 사진에 존재함.
   해당 파일명: GRB003_2022_01_00_001_147160.jpg

한 이미지 내 결함 종류 섞임 정도 
  1종류의 결함이 섞인 이미지: 50,000장 (100.0%)

가장 많이 등장한 결함 조합 
   [crack] 조합 : 40,199장
   [leak] 조합 : 1,887장
   [efflorescence] 조합 : 1,864장
   [detachment] 조합 : 1,625장
   [reticular crack] 조합 : 1,346장

최종 클래스별 이미지 수
   crack : 40,199장
   leak : 1,887장
   efflorescence : 1,864장
   detachment : 1,625장
   reticular crack : 1,346장
   spalling : 1,296장
   material separation : 1,159장
   rebar : 267장
   damage : 243장
   exhilaration : 114장
